# LightGBM

In [ ]:
# Import libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import lightgbm as lgb
from tqdm import tqdm

# Base path for datasets
data_path = "data"

# Load data
receivals = pd.read_csv(f"{data_path}/kernel/receivals.csv")
purchase_orders = pd.read_csv(f"{data_path}/kernel/purchase_orders.csv")
materials = pd.read_csv(f"{data_path}/extended/materials.csv")
transportation = pd.read_csv(f"{data_path}/extended/transportation.csv")

# Convert date columns to datetime
receivals["date_arrival"] = pd.to_datetime(receivals["date_arrival"], utc=True)
purchase_orders["delivery_date"] = pd.to_datetime(purchase_orders["delivery_date"], utc=True)
purchase_orders["created_date_time"] = pd.to_datetime(purchase_orders["created_date_time"], utc=True)
purchase_orders["modified_date_time"] = pd.to_datetime(purchase_orders["modified_date_time"], utc=True)

## Clean and Aggregate Data

In [ ]:
# Remove duplicates
receivals = receivals.drop_duplicates()
purchase_orders = purchase_orders.drop_duplicates()

# Remove physically impossible values
receivals = receivals[receivals["net_weight"] > 0]
purchase_orders = purchase_orders[purchase_orders["quantity"] > 0]

# Standardize units to kilograms
if "unit" in purchase_orders.columns:
    purchase_orders["unit"] = purchase_orders["unit"].str.lower()
    purchase_orders.loc[purchase_orders["unit"].isin(["pund", "lbs", "pound"]), "quantity"] *= 0.45359237
    purchase_orders["unit"] = "kg"

# Ensure temporal consistency (receivals should not happen before order creation)
merged_temp = receivals.merge(
    purchase_orders[["purchase_order_id", "purchase_order_item_no", "created_date_time"]],
    on=["purchase_order_id", "purchase_order_item_no"],
    how="left"
)
receivals = merged_temp[merged_temp["date_arrival"] >= merged_temp["created_date_time"]].copy()
receivals.drop(columns="created_date_time", inplace=True)

# Keep only valid IDs
receivals = receivals[receivals["purchase_order_id"] > 0]
purchase_orders = purchase_orders[purchase_orders["purchase_order_id"] > 0]

# Merge datasets
training_data = pd.merge(
    receivals,
    purchase_orders,
    on=["purchase_order_id", "purchase_order_item_no"],
    how="inner"
)

# Keep only relevant columns for modeling
training_data = training_data[['rm_id', 'date_arrival', 'net_weight', 'quantity']]

# Floor date to day for aggregation
training_data['date_arrival'] = training_data['date_arrival'].dt.floor('D')

# Aggregate data by material and day
training_data = training_data.groupby(['rm_id', 'date_arrival'], as_index=False).agg(
    net_weight=('net_weight', 'sum'),
    quantity=('quantity', 'first')  # TODO: Check later
)
training_data = training_data.sort_values(['rm_id', 'date_arrival'])

# Save aggregated dataset as training_data.csv
training_data.to_csv(f"{data_path}/cleaned_data.csv", index=False)
print(f"✅ cleaning_data.csv created successfully with {len(training_data):,} rows.")

# Optional: Check for missing values
missing_summary = training_data.isnull().sum()
print(missing_summary)

## Feature Engineering 

In [ ]:

data_path = "./data"

# Load aggregated (clean) input
training_data = pd.read_csv(f"{data_path}/cleaned_data.csv")
training_data["date_arrival"] = pd.to_datetime(training_data["date_arrival"], utc=True)

# Build full daily history with zeros for missing days (per rm_id)
full_history_list = []
# Choose the last date you want to cover in the historical file
last_history_date = pd.Timestamp("2024-12-31", tz="UTC")

for rm, group in training_data.groupby("rm_id", sort=False):
    start = group["date_arrival"].min().floor("D")
    full_idx = pd.DataFrame({
        "date_arrival": pd.date_range(start, last_history_date, freq="D", tz="UTC")
    })
    full_idx["rm_id"] = rm
    merged = full_idx.merge(
        group[["rm_id", "date_arrival", "net_weight"]],
        on=["rm_id", "date_arrival"],
        how="left"
    )
    merged["net_weight"] = merged["net_weight"].fillna(0.0)
    full_history_list.append(merged)

full_history = pd.concat(full_history_list, ignore_index=True).sort_values(["rm_id", "date_arrival"])

# -----------------------------
# Add ONLY roll90 (shifted) + weekday dummies dow_0..dow_6
# -----------------------------
def add_features_train(daily_all: pd.DataFrame) -> pd.DataFrame:
    out = []
    ALL_DOWS = [f"dow_{i}" for i in range(7)]  # 0=Mon ... 6=Sun

    for rm, g in daily_all.groupby("rm_id", sort=False):
        g = g.sort_values("date_arrival").copy()
        y = g["net_weight"].astype(float)

        # 90-day avg of *yesterday and earlier* (shift to avoid leakage)
        g["roll90"] = y.shift(1).rolling(90, min_periods=1).mean()

        # Weekday one-hot (guarantee all 7 columns)
        dow = pd.Categorical(g["date_arrival"].dt.weekday, categories=list(range(7)))
        dows = pd.get_dummies(dow, prefix="dow")
        dows = dows.reindex(columns=ALL_DOWS, fill_value=0).reset_index(drop=True)

        g = pd.concat([g.reset_index(drop=True), dows], axis=1)
        out.append(g)

    df = pd.concat(out, ignore_index=True)

    # Clean any NaN/inf (should be rare)
    df["roll90"] = df["roll90"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    for c in ALL_DOWS:
        if c not in df.columns:
            df[c] = 0
        df[c] = df[c].astype(int)

    return df

full_history = add_features_train(full_history)

# Final feature list (used by the model too)
FEATURES = ["roll90"] + [f"dow_{i}" for i in range(7)]

# Keep only relevant columns to train/predict
training_out = full_history[["rm_id", "date_arrival", "net_weight"] + FEATURES]

# Save for the modeling script
training_out.to_csv(f"{data_path}/training_data.csv", index=False)
print(f"✅ training_data.csv created with {len(training_out):,} rows")


## OG Model (gjennombruddet)

In [ ]:
data_path = "./data"

# ----------------------------- Load prepared data ------------------------------
training_data = pd.read_csv(f"{data_path}/training_data.csv")
training_data["date_arrival"] = pd.to_datetime(training_data["date_arrival"], utc=True)

prediction_mapping = pd.read_csv(f"{data_path}/prediction_mapping.csv")
prediction_mapping["forecast_start_date"] = pd.to_datetime(prediction_mapping["forecast_start_date"], utc=True)
prediction_mapping["forecast_end_date"] = pd.to_datetime(prediction_mapping["forecast_end_date"], utc=True)

submission = pd.read_csv(f"{data_path}/sample_submission.csv")

# EXACT same features as preprocessing
FEATURES = ["roll90"] + [f"dow_{i}" for i in range(7)]

# ----------------------------- Train LightGBM ------------------------------
X_train = training_data[FEATURES]
y_train = training_data["net_weight"].astype(float)

model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.2,  # 1.1–1.6 usually good
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=10,
    num_leaves=63,
    n_jobs=-1,
    random_state=42,
    metric="tweedie"
)
model.fit(X_train, y_train)

# ----------------------------- Fast forecasting ------------------------------
# Forecast rule (simple & fast):
# - For each rm_id, use the *last known* roll90 from history.
# - For forecast dates, set roll90 = last_known_roll90 (constant across horizon)
# - Build weekday dummies per date and predict.
# This respects your "only 90-day average + weekday" requirement without slow recursion.

def build_forecast_features(dates_utc: pd.DatetimeIndex, last_roll90: float) -> pd.DataFrame:
    df = pd.DataFrame({"date_arrival": dates_utc})
    df["roll90"] = float(last_roll90)

    # Weekday dummies with guaranteed columns
    dow = pd.Categorical(df["date_arrival"].dt.weekday, categories=list(range(7)))
    dows = pd.get_dummies(dow, prefix="dow")
    dows = dows.reindex(columns=[f"dow_{i}" for i in range(7)], fill_value=0).reset_index(drop=True)

    df = pd.concat([df.reset_index(drop=True), dows], axis=1)
    return df

all_forecasts = []

for _, row in tqdm(prediction_mapping.iterrows(), total=len(prediction_mapping), desc="Forecasting RM IDs"):
    rm_id = row["rm_id"]
    ID = row["ID"]
    dates = pd.date_range(row["forecast_start_date"], row["forecast_end_date"], freq="D", tz="UTC")

    # Historical rows for this rm
    hist = training_data.loc[training_data["rm_id"] == rm_id].sort_values("date_arrival")

    if hist.empty:
        last_roll90 = 0.0
    else:
        # Use the *last known* roll90 from history
        r90 = hist["roll90"].dropna()
        last_roll90 = float(r90.iloc[-1]) if len(r90) else 0.0

    feats = build_forecast_features(dates, last_roll90)
    X_group = feats[FEATURES]
    preds = model.predict(X_group)
    preds = np.clip(preds, 0.0, None)

    out = pd.DataFrame({
        "ID": ID,
        "rm_id": rm_id,
        "date_arrival": dates,
        "net_weight_pred": preds
    })
    all_forecasts.append(out[["ID", "rm_id", "date_arrival", "net_weight_pred"]])

forecast_df = pd.concat(all_forecasts, ignore_index=True)

# ----------------------------- Aggregate & export ------------------------------
agg_predictions = forecast_df.groupby("ID", as_index=False)["net_weight_pred"].sum()
submission = submission[["ID"]].merge(agg_predictions, on="ID", how="left")
submission["predicted_weight"] = submission["net_weight_pred"].fillna(0.0)
submission = submission[["ID", "predicted_weight"]]
submission.to_csv(f"{data_path}/submission_lgbm4.csv", index=False)
print(f"✅ submission_lgbm4.csv created with {len(submission):,} rows")


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("data")
submission_path = data_path / "submission_lgbm3.csv"   # <-- your file
mapping_path = data_path / "prediction_mapping.csv"
rm_no2024_path = data_path / "zero_materials.csv"

out_path = submission_path.with_name(submission_path.stem + "_zeroed.csv")

submission = pd.read_csv(submission_path)
mapping = pd.read_csv(mapping_path)
rm_no2024 = pd.read_csv(rm_no2024_path)

mapping["ID"] = mapping["ID"].astype(str)
submission["ID"] = submission["ID"].astype(str)
mapping["rm_id"] = pd.to_numeric(mapping["rm_id"], errors="coerce")
rm_no2024["rm_id"] = pd.to_numeric(rm_no2024["rm_id"], errors="coerce").astype("Int64")
rm_set = set(rm_no2024["rm_id"].dropna().astype(int).tolist())

sub_m = submission.merge(mapping[["ID","rm_id"]], on="ID", how="left")
to_zero = sub_m["rm_id"].isin(rm_set)

if "predicted_weight" not in sub_m.columns:
    sub_m["predicted_weight"] = 0.0  # if your file only had ID

sub_m.loc[to_zero, "predicted_weight"] = 0.0

out = sub_m[["ID","predicted_weight"]]
out.to_csv(out_path, index=False)

affected = sub_m.loc[to_zero, ["ID","rm_id"]].drop_duplicates().sort_values(["rm_id","ID"])

print("Saved:", out_path)


## K-means Clustering k=3 (5896)

In [ ]:
# model_kmeans_clusters.py
# -------------------------------------------------------------------------
# KMeans cluster materials by simple series stats, train one LGBM per cluster,
# and fall back to a global model when needed. Keeps features identical to your
# current pipeline: roll90 + weekday dummies.
# -------------------------------------------------------------------------
import pandas as pd
import numpy as np
import lightgbm as lgb
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# ----------------------------- Config -----------------------------
DATA_PATH = Path("./data")
TRAIN_PATH = DATA_PATH / "training_data.csv"
MAP_PATH   = DATA_PATH / "prediction_mapping.csv"
SUB_PATH   = DATA_PATH / "sample_submission.csv"
OUT_PATH   = DATA_PATH / "submission_lgbm5_kmeans.csv"

# Cluster count (try 2–5)
K = 3

# ----------------------------- Load data -----------------------------
training_data = pd.read_csv(TRAIN_PATH)
training_data["date_arrival"] = pd.to_datetime(training_data["date_arrival"], utc=True)
# Normalize rm_id (consistent int dtype; allow missing as NA)
training_data["rm_id"] = pd.to_numeric(training_data["rm_id"], errors="coerce").astype("Int64")

prediction_mapping = pd.read_csv(MAP_PATH)
prediction_mapping["forecast_start_date"] = pd.to_datetime(prediction_mapping["forecast_start_date"], utc=True)
prediction_mapping["forecast_end_date"]   = pd.to_datetime(prediction_mapping["forecast_end_date"],   utc=True)

submission = pd.read_csv(SUB_PATH)

# EXACT same features as your pipeline
FEATURES = ["roll90"] + [f"dow_{i}" for i in range(7)]

# Sanity checks
missing_cols = [c for c in FEATURES + ["net_weight","rm_id","date_arrival"] if c not in training_data.columns]
if missing_cols:
    raise ValueError(f"Missing columns in training_data.csv: {missing_cols}")

# ----------------------------- Cluster features per rm_id -----------------------------
# Cheap, robust summary statistics per material
g = training_data.sort_values(["rm_id", "date_arrival"]).copy()

def _last_non_nan(s):
    s = s.dropna()
    return float(s.iloc[-1]) if len(s) else 0.0

per_rm = (
    g.groupby("rm_id", dropna=True)
     .agg(
         mean_weight=("net_weight", "mean"),
         pct_nonzero=("net_weight", lambda s: float((s > 0).mean())),
         last_roll90=("roll90", _last_non_nan),
     )
     .reset_index()
)
per_rm["rm_id"] = per_rm["rm_id"].astype("Int64")

# Handle potential empty per_rm
if per_rm.empty:
    raise ValueError("No per-rm_id statistics available. Check that training_data has rows and rm_id values.")

# Standardize & KMeans
cluster_feats = per_rm[["mean_weight", "pct_nonzero", "last_roll90"]].fillna(0.0).astype(float)
scaler = StandardScaler()
Xc = scaler.fit_transform(cluster_feats)

kmeans = KMeans(n_clusters=K, n_init=20, random_state=42)
per_rm["cluster"] = kmeans.fit_predict(Xc)
rm2cluster = dict(zip(per_rm["rm_id"].astype(int), per_rm["cluster"].astype(int)))

print("Cluster sizes:", per_rm["cluster"].value_counts().sort_index().to_dict())

# ----------------------------- Train cluster models + global fallback -----------------------------
def make_lgbm():
    return lgb.LGBMRegressor(
        objective="tweedie",
        tweedie_variance_power=1.2,  # try 1.1–1.6
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=10,
        num_leaves=63,
        n_jobs=-1,
        random_state=42,
        metric="tweedie"
    )

cluster_models = {}
for c in range(K):
    rm_ids_c = set(per_rm.loc[per_rm["cluster"] == c, "rm_id"].dropna().astype(int).tolist())
    df_c = training_data[training_data["rm_id"].isin(rm_ids_c)]
    if df_c.empty:
        print(f"[Warn] Cluster {c} has no rows; will use global fallback for it.")
        continue
    X_train = df_c[FEATURES]
    y_train = df_c["net_weight"].astype(float)
    m = make_lgbm()
    m.fit(X_train, y_train)
    cluster_models[c] = m
    print(f"Trained cluster {c}: {len(df_c):,} rows, {len(rm_ids_c):,} rm_id")

# Always train a global fallback model on everything
X_all = training_data[FEATURES]
y_all = training_data["net_weight"].astype(float)
m_global = make_lgbm()
m_global.fit(X_all, y_all)
cluster_models["global"] = m_global

# ----------------------------- Forecast features -----------------------------
def build_forecast_features(dates_utc: pd.DatetimeIndex, last_roll90: float) -> pd.DataFrame:
    df = pd.DataFrame({"date_arrival": dates_utc})
    df["roll90"] = float(last_roll90)
    # Weekday dummies with guaranteed columns
    dow = pd.Categorical(df["date_arrival"].dt.weekday, categories=list(range(7)))
    dows = pd.get_dummies(dow, prefix="dow")
    dows = dows.reindex(columns=[f"dow_{i}" for i in range(7)], fill_value=0).reset_index(drop=True)
    return pd.concat([df.reset_index(drop=True), dows], axis=1)

# ----------------------------- Forecast all IDs -----------------------------
all_forecasts = []

for _, row in tqdm(prediction_mapping.iterrows(), total=len(prediction_mapping), desc="Forecasting RM IDs"):
    rm_id_val = pd.to_numeric(row["rm_id"], errors="coerce")
    rm_id_int = int(rm_id_val) if pd.notna(rm_id_val) else None
    ID = row["ID"]
    dates = pd.date_range(row["forecast_start_date"], row["forecast_end_date"], freq="D", tz="UTC")

    # Choose cluster model (with global fallback)
    c = rm2cluster.get(rm_id_int, None)
    m = cluster_models.get(c, cluster_models["global"])

    # Last known roll90 from history (static rule)
    hist = training_data.loc[training_data["rm_id"] == rm_id_int].sort_values("date_arrival")
    if hist.empty:
        last_roll90 = 0.0
    else:
        r90 = hist["roll90"].dropna()
        last_roll90 = float(r90.iloc[-1]) if len(r90) else 0.0

    feats = build_forecast_features(dates, last_roll90)
    preds = m.predict(feats[FEATURES])
    preds = np.clip(preds, 0.0, None)

    out = pd.DataFrame({
        "ID": ID,
        "rm_id": rm_id_int,
        "date_arrival": dates,
        "net_weight_pred": preds
    })
    all_forecasts.append(out[["ID", "rm_id", "date_arrival", "net_weight_pred"]])

forecast_df = pd.concat(all_forecasts, ignore_index=True)

# ----------------------------- Aggregate & export -----------------------------
agg_predictions = forecast_df.groupby("ID", as_index=False)["net_weight_pred"].sum()
submission = submission[["ID"]].merge(agg_predictions, on="ID", how="left")
submission["predicted_weight"] = submission["net_weight_pred"].fillna(0.0)
submission = submission[["ID", "predicted_weight"]]
submission.to_csv(OUT_PATH, index=False)
print(f"✅ {OUT_PATH.name} created with {len(submission):,} rows")


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("data")
submission_path = data_path / "submission_lgbm5_kmeans.csv"   # <-- your file
mapping_path = data_path / "prediction_mapping.csv"
rm_no2024_path = data_path / "zero_materials.csv"

out_path = submission_path.with_name(submission_path.stem + "_zeroedKmeans.csv")

submission = pd.read_csv(submission_path)
mapping = pd.read_csv(mapping_path)
rm_no2024 = pd.read_csv(rm_no2024_path)

mapping["ID"] = mapping["ID"].astype(str)
submission["ID"] = submission["ID"].astype(str)
mapping["rm_id"] = pd.to_numeric(mapping["rm_id"], errors="coerce")
rm_no2024["rm_id"] = pd.to_numeric(rm_no2024["rm_id"], errors="coerce").astype("Int64")
rm_set = set(rm_no2024["rm_id"].dropna().astype(int).tolist())

sub_m = submission.merge(mapping[["ID","rm_id"]], on="ID", how="left")
to_zero = sub_m["rm_id"].isin(rm_set)


if "predicted_weight" not in sub_m.columns:
    sub_m["predicted_weight"] = 0.0  # if your file only had ID

sub_m.loc[to_zero, "predicted_weight"] = 0.0

out = sub_m[["ID","predicted_weight"]]
out.to_csv(out_path, index=False)

affected = sub_m.loc[to_zero, ["ID","rm_id"]].drop_duplicates().sort_values(["rm_id","ID"])

print("Saved:", out_path)


## K-Means Clustering + K-Fold Cross Validation (5625)

In [ ]:
# model_kmeans_clusters.py
# -------------------------------------------------------------------------
# KMeans cluster materials by simple series stats, train one LightGBM K-fold
# ensemble per cluster, and fall back to a global K-fold ensemble when needed.
# Keeps features identical to your current pipeline: roll90 + weekday dummies.
# -------------------------------------------------------------------------
import pandas as pd
import numpy as np
import lightgbm as lgb
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold  # swap to GroupKFold/TimeSeriesSplit if desired
from pathlib import Path

# ----------------------------- Config -----------------------------
DATA_PATH = Path("./data")
TRAIN_PATH = DATA_PATH / "training_data.csv"
MAP_PATH   = DATA_PATH / "prediction_mapping.csv"
SUB_PATH   = DATA_PATH / "sample_submission.csv"
OUT_PATH   = DATA_PATH / "submission_lgbm5_kmeans_kfold.csv"

# Cluster count (your best so far)
K = 3

# K-Fold settings
N_FOLDS = 5          # target number of folds
EARLY_STOP = 200     # early stopping rounds (None to disable)
VERBOSE = 0          # LightGBM early_stopping verbosity
SEED = 42

# EXACT same features as your pipeline
FEATURES = ["roll90"] + [f"dow_{i}" for i in range(7)]

# ----------------------------- Load data -----------------------------
training_data = pd.read_csv(TRAIN_PATH)
training_data["date_arrival"] = pd.to_datetime(training_data["date_arrival"], utc=True)
# Normalize rm_id (consistent int dtype; allow missing as NA)
training_data["rm_id"] = pd.to_numeric(training_data["rm_id"], errors="coerce").astype("Int64")

prediction_mapping = pd.read_csv(MAP_PATH)
prediction_mapping["forecast_start_date"] = pd.to_datetime(prediction_mapping["forecast_start_date"], utc=True)
prediction_mapping["forecast_end_date"]   = pd.to_datetime(prediction_mapping["forecast_end_date"],   utc=True)

submission = pd.read_csv(SUB_PATH)

# Sanity checks
missing_cols = [c for c in FEATURES + ["net_weight","rm_id","date_arrival"] if c not in training_data.columns]
if missing_cols:
    raise ValueError(f"Missing columns in training_data.csv: {missing_cols}")

# ----------------------------- Cluster features per rm_id -----------------------------
# Cheap, robust summary statistics per material
g = training_data.sort_values(["rm_id", "date_arrival"]).copy()

def _last_non_nan(s: pd.Series) -> float:
    s = s.dropna()
    return float(s.iloc[-1]) if len(s) else 0.0

per_rm = (
    g.groupby("rm_id", dropna=True)
     .agg(
         mean_weight=("net_weight", "mean"),
         pct_nonzero=("net_weight", lambda s: float((s > 0).mean())),
         last_roll90=("roll90", _last_non_nan),
     )
     .reset_index()
)
per_rm["rm_id"] = per_rm["rm_id"].astype("Int64")

# Handle potential empty per_rm
if per_rm.empty:
    raise ValueError("No per-rm_id statistics available. Check that training_data has rows and rm_id values.")

# Standardize & KMeans
cluster_feats = per_rm[["mean_weight", "pct_nonzero", "last_roll90"]].fillna(0.0).astype(float)
scaler = StandardScaler()
Xc = scaler.fit_transform(cluster_feats)

kmeans = KMeans(n_clusters=K, n_init=20, random_state=SEED)
per_rm["cluster"] = kmeans.fit_predict(Xc)
rm2cluster = dict(zip(per_rm["rm_id"].dropna().astype(int), per_rm["cluster"].astype(int)))

print("Cluster sizes:", per_rm["cluster"].value_counts().sort_index().to_dict())

# ----------------------------- LGBM helpers -----------------------------
def make_lgbm() -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(
        objective="tweedie",
        tweedie_variance_power=1.2,  # try 1.1–1.6
        n_estimators=5000,
        learning_rate=0.05,
        max_depth=10,
        num_leaves=63,
        n_jobs=-1,
        random_state=SEED,
        metric="tweedie",
        verbosity=-1,
    )

def _folds_for(n_rows: int, target_folds: int = N_FOLDS) -> int:
    """
    Choose a safe number of folds for a dataset with n_rows rows.
    Ensures at least 2 folds and at most n_rows-1 (KFold requirement),
    and avoids super-tiny validation splits by capping to roughly n/5.
    """
    if n_rows < 4:
        return 2  # minimal but still allows a val split
    cap_by_rows = max(2, min(target_folds, n_rows - 1))
    cap_by_size = max(2, min(cap_by_rows, n_rows // 5 if n_rows // 5 >= 2 else cap_by_rows))
    return cap_by_size

def train_kfold_models(X: pd.DataFrame, y: pd.Series):
    """Train K-fold LightGBM models (using pandas throughout) and return list of models."""
    n_rows = len(X)
    n_splits = _folds_for(n_rows, N_FOLDS)
    if n_splits < 2:
        # Fallback: train a single model if we can't make a proper split
        m = make_lgbm()
        m.fit(X, y.astype(float))
        print(f"  (rows={n_rows}) insufficient for CV; trained single model")
        return [m]

    models = []
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    yv = y.astype(float)
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        m = make_lgbm()
        X_tr, y_tr = X.iloc[tr_idx], yv.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], yv.iloc[va_idx]
        if EARLY_STOP:
            m.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                eval_metric="tweedie",
                callbacks=[lgb.early_stopping(EARLY_STOP, verbose=VERBOSE)],
            )
        else:
            m.fit(X_tr, y_tr)
        models.append(m)
        best_iter = getattr(m, "best_iteration_", None)
        bi = f" (best_iter={best_iter})" if best_iter else ""
        print(f"  fold {fold}/{n_splits} trained{bi}")
    return models

def predict_mean(models, X: pd.DataFrame) -> np.ndarray:
    """Average predictions across fold models (use best_iteration_ when available)."""
    preds = []
    for m in models:
        kw = {}
        if getattr(m, "best_iteration_", None):
            kw["num_iteration"] = m.best_iteration_
        # Keep DataFrame so LightGBM sees feature names; ensure column order == FEATURES
        preds.append(m.predict(X, **kw))
    return np.mean(np.vstack(preds), axis=0)

# ----------------------------- Train cluster models + global fallback -----------------------------
cluster_models: dict = {}
for c in range(K):
    rm_ids_c = set(per_rm.loc[per_rm["cluster"] == c, "rm_id"].dropna().astype(int).tolist())
    df_c = training_data[training_data["rm_id"].isin(rm_ids_c)]
    if df_c.empty:
        print(f"[Warn] Cluster {c} has no rows; will use global fallback for it.")
        continue
    X_train = df_c[FEATURES]
    y_train = df_c["net_weight"].astype(float)
    print(f"Training CV models for cluster {c}: {len(df_c):,} rows, {len(rm_ids_c):,} rm_id")
    models_c = train_kfold_models(X_train, y_train)
    cluster_models[c] = models_c

# Always train a global fallback model on everything
X_all = training_data[FEATURES]
y_all = training_data["net_weight"].astype(float)
print(f"Training CV global fallback on all data: {len(training_data):,} rows")
cluster_models["global"] = train_kfold_models(X_all, y_all)

# ----------------------------- Forecast features -----------------------------
def build_forecast_features(dates_utc: pd.DatetimeIndex, last_roll90: float) -> pd.DataFrame:
    df = pd.DataFrame({"date_arrival": dates_utc})
    df["roll90"] = float(last_roll90)
    # Weekday dummies with guaranteed columns
    dow = pd.Categorical(df["date_arrival"].dt.weekday, categories=list(range(7)))
    dows = pd.get_dummies(dow, prefix="dow")
    dows = dows.reindex(columns=[f"dow_{i}" for i in range(7)], fill_value=0).reset_index(drop=True)
    feats = pd.concat([df.reset_index(drop=True), dows], axis=1)
    # return only model features in the right order for LightGBM
    return feats.reindex(columns=["date_arrival"] + FEATURES, fill_value=0)

# ----------------------------- Forecast all IDs -----------------------------
all_forecasts = []

for _, row in tqdm(prediction_mapping.iterrows(), total=len(prediction_mapping), desc="Forecasting RM IDs"):
    rm_id_val = pd.to_numeric(row["rm_id"], errors="coerce")
    rm_id_int = int(rm_id_val) if pd.notna(rm_id_val) else None
    ID = row["ID"]
    dates = pd.date_range(row["forecast_start_date"], row["forecast_end_date"], freq="D", tz="UTC")

    # Choose cluster model (with global fallback)
    c = rm2cluster.get(rm_id_int, None)
    models = cluster_models.get(c, cluster_models["global"])

    # Last known roll90 from history (static rule)
    hist = training_data.loc[training_data["rm_id"] == rm_id_int].sort_values("date_arrival")
    if hist.empty:
        last_roll90 = 0.0
    else:
        r90 = hist["roll90"].dropna()
        last_roll90 = float(r90.iloc[-1]) if len(r90) else 0.0

    feats_full = build_forecast_features(dates, last_roll90)
    # Strictly pass only FEATURES (ordered) to the models
    featsX = feats_full[FEATURES]
    preds = predict_mean(models, featsX)
    preds = np.clip(preds, 0.0, None)

    out = pd.DataFrame({
        "ID": ID,
        "rm_id": rm_id_int,
        "date_arrival": dates,
        "net_weight_pred": preds
    })
    all_forecasts.append(out[["ID", "rm_id", "date_arrival", "net_weight_pred"]])

forecast_df = pd.concat(all_forecasts, ignore_index=True)

# ----------------------------- Aggregate & export -----------------------------
agg_predictions = forecast_df.groupby("ID", as_index=False)["net_weight_pred"].sum()
submission = submission[["ID"]].merge(agg_predictions, on="ID", how="left")
submission["predicted_weight"] = submission["net_weight_pred"].fillna(0.0)
submission = submission[["ID", "predicted_weight"]]
submission.to_csv(OUT_PATH, index=False)
print(f"✅ {OUT_PATH.name} created with {len(submission):,} rows")

# ----------------------------- Notes -----------------------------
# - To prevent cross-material leakage inside a cluster, swap KFold for GroupKFold
#   with groups=df_c["rm_id"] (and groups=training_data["rm_id"] for the global model).
# - For strictly temporal validation, use TimeSeriesSplit; make sure your rows are
#   sorted by date_arrival before splitting, and be aware TSS has no shuffling.
# - Tune N_FOLDS, EARLY_STOP, and LGBM params as needed without changing pipeline shape.


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("data")
submission_path = data_path / "submission_lgbm5_kmeans_kfold.csv"   # <-- your file
mapping_path = data_path / "prediction_mapping.csv"
rm_no2024_path = data_path / "zero_materials.csv"

out_path = submission_path.with_name(submission_path.stem + "_zeroed.csv")

submission = pd.read_csv(submission_path)
mapping = pd.read_csv(mapping_path)
rm_no2024 = pd.read_csv(rm_no2024_path)

mapping["ID"] = mapping["ID"].astype(str)
submission["ID"] = submission["ID"].astype(str)
mapping["rm_id"] = pd.to_numeric(mapping["rm_id"], errors="coerce")
rm_no2024["rm_id"] = pd.to_numeric(rm_no2024["rm_id"], errors="coerce").astype("Int64")
rm_set = set(rm_no2024["rm_id"].dropna().astype(int).tolist())

sub_m = submission.merge(mapping[["ID","rm_id"]], on="ID", how="left")
to_zero = sub_m["rm_id"].isin(rm_set)

if "predicted_weight" not in sub_m.columns:
    sub_m["predicted_weight"] = 0.0  # if your file only had ID

sub_m.loc[to_zero, "predicted_weight"] = 0.0

out = sub_m[["ID","predicted_weight"]]
out.to_csv(out_path, index=False)

affected = sub_m.loc[to_zero, ["ID","rm_id"]].drop_duplicates().sort_values(["rm_id","ID"])

print("Saved:", out_path)


## K-Means Clustering + K-Fold Cross Validation + Hyperparameter Tuning (5341)

In [ ]:
# model_kmeans_clusters.py
# -------------------------------------------------------------------------
# KMeans cluster materials by simple series stats, train one LightGBM K-fold
# ensemble per cluster, and fall back to a global K-fold ensemble when needed.
# Optional: lightweight global hyperparameter tuning reused for all models.
# Keeps features identical to your pipeline: roll90 + weekday dummies.
# -------------------------------------------------------------------------
import pandas as pd
import numpy as np
import lightgbm as lgb
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold  # swap to GroupKFold/TimeSeriesSplit if desired
from sklearn.metrics import mean_tweedie_deviance
from pathlib import Path
from typing import Dict, Any

# ----------------------------- Config -----------------------------
DATA_PATH = Path("./data")
TRAIN_PATH = DATA_PATH / "training_data.csv"
MAP_PATH   = DATA_PATH / "prediction_mapping.csv"
SUB_PATH   = DATA_PATH / "sample_submission.csv"
OUT_PATH   = DATA_PATH / "submission_lgbm5_kmeans_kfold_tuned.csv"

# Clustering
K = 3  # your best so far

# K-Fold (final training)
N_FOLDS = 5
EARLY_STOP = 200     # early stopping rounds (None to disable)
VERBOSE = 0          # LightGBM early_stopping verbosity
SEED = 42

# Optional hyperparameter tuning (global once, reused for all clusters)
DO_TUNING = True
TUNE_TRIALS = 20            # total random candidates
TUNE_FOLDS = 3              # CV folds during tuning
TUNE_SUBSAMPLE_N = 200_000  # rows to sample for tuning (None = use all)
TUNE_N_ESTIMATORS = 1000    # faster fixed trees during tuning (no early stopping)

# EXACT same features as your pipeline
FEATURES = ["roll90"] + [f"dow_{i}" for i in range(7)]

# ----------------------------- Load data -----------------------------
training_data = pd.read_csv(TRAIN_PATH)
training_data["date_arrival"] = pd.to_datetime(training_data["date_arrival"], utc=True)
# Normalize rm_id (consistent int dtype; allow missing as NA)
training_data["rm_id"] = pd.to_numeric(training_data["rm_id"], errors="coerce").astype("Int64")

prediction_mapping = pd.read_csv(MAP_PATH)
prediction_mapping["forecast_start_date"] = pd.to_datetime(prediction_mapping["forecast_start_date"], utc=True)
prediction_mapping["forecast_end_date"]   = pd.to_datetime(prediction_mapping["forecast_end_date"],   utc=True)

submission = pd.read_csv(SUB_PATH)

# Sanity checks
missing_cols = [c for c in FEATURES + ["net_weight","rm_id","date_arrival"] if c not in training_data.columns]
if missing_cols:
    raise ValueError(f"Missing columns in training_data.csv: {missing_cols}")

# ----------------------------- Cluster features per rm_id -----------------------------
g = training_data.sort_values(["rm_id", "date_arrival"]).copy()

def _last_non_nan(s: pd.Series) -> float:
    s = s.dropna()
    return float(s.iloc[-1]) if len(s) else 0.0

per_rm = (
    g.groupby("rm_id", dropna=True)
     .agg(
         mean_weight=("net_weight", "mean"),
         pct_nonzero=("net_weight", lambda s: float((s > 0).mean())),
         last_roll90=("roll90", _last_non_nan),
     )
     .reset_index()
)
per_rm["rm_id"] = per_rm["rm_id"].astype("Int64")

if per_rm.empty:
    raise ValueError("No per-rm_id statistics available. Check that training_data has rows and rm_id values.")

# Standardize & KMeans
cluster_feats = per_rm[["mean_weight", "pct_nonzero", "last_roll90"]].fillna(0.0).astype(float)
scaler = StandardScaler()
Xc = scaler.fit_transform(cluster_feats)

kmeans = KMeans(n_clusters=K, n_init=20, random_state=SEED)
per_rm["cluster"] = kmeans.fit_predict(Xc)
rm2cluster = dict(zip(per_rm["rm_id"].dropna().astype(int), per_rm["cluster"].astype(int)))

print("Cluster sizes:", per_rm["cluster"].value_counts().sort_index().to_dict())

# ----------------------------- Hyperparameter tuning -----------------------------
# Base params (used if DO_TUNING=False; also seed/metric for tuned models)
BASE_PARAMS: Dict[str, Any] = dict(
    objective="tweedie",
    tweedie_variance_power=1.2,  # try 1.1–1.6
    n_estimators=5000,
    learning_rate=0.05,
    max_depth=10,
    num_leaves=63,
    n_jobs=-1,
    random_state=SEED,
    metric="tweedie",
    verbosity=-1,
)

def _rand_float(lo, hi):
    return np.random.RandomState(SEED).uniform(lo, hi) if lo == hi else np.random.uniform(lo, hi)

def _rand_int(lo, hi):
    return np.random.randint(lo, hi + 1) if hi > lo else lo

def sample_candidate() -> Dict[str, Any]:
    # Reasonable LightGBM space for Tweedie; bounded to avoid extremes
    return {
        "learning_rate": 10 ** np.random.uniform(np.log10(0.01), np.log10(0.2)),
        "num_leaves": _rand_int(31, 255),
        "max_depth": np.random.choice([-1, 6, 8, 10, 12, 14, 16]),
        "feature_fraction": _rand_float(0.6, 1.0),
        "bagging_fraction": _rand_float(0.6, 1.0),
        "bagging_freq": _rand_int(1, 7),
        "min_child_samples": _rand_int(10, 200),
        "lambda_l1": _rand_float(0.0, 2.0),
        "lambda_l2": _rand_float(0.0, 2.0),
        "tweedie_variance_power": _rand_float(1.1, 1.6),
    }

def tune_global_params(X: pd.DataFrame, y: pd.Series) -> Dict[str, Any]:
    rng = np.random.RandomState(SEED)
    if TUNE_SUBSAMPLE_N is not None and len(X) > TUNE_SUBSAMPLE_N:
        idx = rng.choice(len(X), size=TUNE_SUBSAMPLE_N, replace=False)
        Xs, ys = X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
        print(f"Tuning on subsample: {len(Xs):,} rows")
    else:
        Xs, ys = X.reset_index(drop=True), y.reset_index(drop=True)
        print(f"Tuning on full data: {len(Xs):,} rows")

    kf = KFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=SEED)

    best_score = np.inf
    best_params = {}

    for t in range(1, TUNE_TRIALS + 1):
        cand = sample_candidate()
        # Fixed trees & no early stopping during tuning for fairness/speed
        params = {
            **BASE_PARAMS,
            **cand,
            "n_estimators": TUNE_N_ESTIMATORS,
        }

        cv_scores = []
        for tr_idx, va_idx in kf.split(Xs):
            m = lgb.LGBMRegressor(**params)
            m.fit(Xs.iloc[tr_idx], ys.iloc[tr_idx])
            preds = m.predict(Xs.iloc[va_idx])
            # Tweedie deviance lower is better (consistent with LightGBM metric)
            cv_scores.append(mean_tweedie_deviance(ys.iloc[va_idx], preds, power=params["tweedie_variance_power"]))
        score = float(np.mean(cv_scores))

        print(f"  trial {t:02d}/{TUNE_TRIALS} score={score:.6f} params={ {k: cand[k] for k in cand} }")

        if score < best_score:
            best_score = score
            best_params = cand

    print("Best tuning score:", best_score)
    print("Best params:", best_params)
    # Merge best into base for final training (with early stopping & 5-fold)
    tuned = {**BASE_PARAMS, **best_params}
    return tuned

# ----------------------------- LGBM helpers -----------------------------
def make_lgbm(final_params: Dict[str, Any]) -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(**final_params)

def _folds_for(n_rows: int, target_folds: int = N_FOLDS) -> int:
    if n_rows < 4:
        return 2
    cap_by_rows = max(2, min(target_folds, n_rows - 1))
    cap_by_size = max(2, min(cap_by_rows, n_rows // 5 if n_rows // 5 >= 2 else cap_by_rows))
    return cap_by_size

def train_kfold_models(X: pd.DataFrame, y: pd.Series, final_params: Dict[str, Any]):
    """Train K-fold LightGBM models (pandas throughout) and return list of models."""
    n_rows = len(X)
    n_splits = _folds_for(n_rows, N_FOLDS)
    if n_splits < 2:
        m = make_lgbm(final_params)
        m.fit(X, y.astype(float))
        print(f"  (rows={n_rows}) insufficient for CV; trained single model")
        return [m]

    models = []
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    yv = y.astype(float)
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        m = make_lgbm(final_params)
        X_tr, y_tr = X.iloc[tr_idx], yv.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], yv.iloc[va_idx]
        if EARLY_STOP:
            m.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                eval_metric="tweedie",
                callbacks=[lgb.early_stopping(EARLY_STOP, verbose=VERBOSE)],
            )
        else:
            m.fit(X_tr, y_tr)
        models.append(m)
        best_iter = getattr(m, "best_iteration_", None)
        bi = f" (best_iter={best_iter})" if best_iter else ""
        print(f"  fold {fold}/{n_splits} trained{bi}")
    return models

def predict_mean(models, X: pd.DataFrame) -> np.ndarray:
    preds = []
    for m in models:
        kw = {}
        if getattr(m, "best_iteration_", None):
            kw["num_iteration"] = m.best_iteration_
        preds.append(m.predict(X, **kw))
    return np.mean(np.vstack(preds), axis=0)

# ----------------------------- Train cluster models + global fallback -----------------------------
# Optionally tune once globally, reuse for all training
X_all = training_data[FEATURES]
y_all = training_data["net_weight"].astype(float)

if DO_TUNING:
    print(">>> Hyperparameter tuning enabled")
    FINAL_PARAMS = tune_global_params(X_all, y_all)
else:
    print(">>> Hyperparameter tuning disabled (using BASE_PARAMS)")
    FINAL_PARAMS = BASE_PARAMS

# Ensure final training uses your larger n_estimators (and any tuned values)
FINAL_PARAMS = {**FINAL_PARAMS, "n_estimators": BASE_PARAMS["n_estimators"]}

cluster_models: dict = {}
for c in range(K):
    rm_ids_c = set(per_rm.loc[per_rm["cluster"] == c, "rm_id"].dropna().astype(int).tolist())
    df_c = training_data[training_data["rm_id"].isin(rm_ids_c)]
    if df_c.empty:
        print(f"[Warn] Cluster {c} has no rows; will use global fallback for it.")
        continue
    X_train = df_c[FEATURES]
    y_train = df_c["net_weight"].astype(float)
    print(f"Training CV models for cluster {c}: {len(df_c):,} rows, {len(rm_ids_c):,} rm_id")
    models_c = train_kfold_models(X_train, y_train, FINAL_PARAMS)
    cluster_models[c] = models_c

# Always train a global fallback model on everything
print(f"Training CV global fallback on all data: {len(training_data):,} rows")
cluster_models["global"] = train_kfold_models(X_all, y_all, FINAL_PARAMS)

# ----------------------------- Forecast features -----------------------------
def build_forecast_features(dates_utc: pd.DatetimeIndex, last_roll90: float) -> pd.DataFrame:
    df = pd.DataFrame({"date_arrival": dates_utc})
    df["roll90"] = float(last_roll90)
    dow = pd.Categorical(df["date_arrival"].dt.weekday, categories=list(range(7)))
    dows = pd.get_dummies(dow, prefix="dow")
    dows = dows.reindex(columns=[f"dow_{i}" for i in range(7)], fill_value=0).reset_index(drop=True)
    feats = pd.concat([df.reset_index(drop=True), dows], axis=1)
    return feats.reindex(columns=["date_arrival"] + FEATURES, fill_value=0)

# ----------------------------- Forecast all IDs -----------------------------
all_forecasts = []

for _, row in tqdm(prediction_mapping.iterrows(), total=len(prediction_mapping), desc="Forecasting RM IDs"):
    rm_id_val = pd.to_numeric(row["rm_id"], errors="coerce")
    rm_id_int = int(rm_id_val) if pd.notna(rm_id_val) else None
    ID = row["ID"]
    dates = pd.date_range(row["forecast_start_date"], row["forecast_end_date"], freq="D", tz="UTC")

    # Choose cluster model (with global fallback)
    c = rm2cluster.get(rm_id_int, None)
    models = cluster_models.get(c, cluster_models["global"])

    # Last known roll90 from history (static rule)
    hist = training_data.loc[training_data["rm_id"] == rm_id_int].sort_values("date_arrival")
    if hist.empty:
        last_roll90 = 0.0
    else:
        r90 = hist["roll90"].dropna()
        last_roll90 = float(r90.iloc[-1]) if len(r90) else 0.0

    feats_full = build_forecast_features(dates, last_roll90)
    featsX = feats_full[FEATURES]
    preds = predict_mean(models, featsX)
    preds = np.clip(preds, 0.0, None)

    out = pd.DataFrame({
        "ID": ID,
        "rm_id": rm_id_int,
        "date_arrival": dates,
        "net_weight_pred": preds
    })
    all_forecasts.append(out[["ID", "rm_id", "date_arrival", "net_weight_pred"]])

forecast_df = pd.concat(all_forecasts, ignore_index=True)

# ----------------------------- Aggregate & export -----------------------------
agg_predictions = forecast_df.groupby("ID", as_index=False)["net_weight_pred"].sum()
submission = submission[["ID"]].merge(agg_predictions, on="ID", how="left")
submission["predicted_weight"] = submission["net_weight_pred"].fillna(0.0)
submission = submission[["ID", "predicted_weight"]]
submission.to_csv(OUT_PATH, index=False)
print(f"✅ {OUT_PATH.name} created with {len(submission):,} rows")

# ----------------------------- Notes -----------------------------
# - Want leakage-safe CV? Use GroupKFold with groups=df_c["rm_id"] for clusters
#   and groups=training_data["rm_id"] for global.
# - Want temporal CV? Use TimeSeriesSplit on date-sorted rows (no shuffling).
# - To disable tuning, set DO_TUNING=False (keeps fixed BASE_PARAMS).
# - You can widen/narrow the search space in sample_candidate().


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("data")
submission_path = data_path / "submission_lgbm5_kmeans_kfold_tuned.csv"   # <-- your file
mapping_path = data_path / "prediction_mapping.csv"
rm_no2024_path = data_path / "zero_materials.csv"

out_path = submission_path.with_name(submission_path.stem + "_zeroed.csv")

submission = pd.read_csv(submission_path)
mapping = pd.read_csv(mapping_path)
rm_no2024 = pd.read_csv(rm_no2024_path)

mapping["ID"] = mapping["ID"].astype(str)
submission["ID"] = submission["ID"].astype(str)
mapping["rm_id"] = pd.to_numeric(mapping["rm_id"], errors="coerce")
rm_no2024["rm_id"] = pd.to_numeric(rm_no2024["rm_id"], errors="coerce").astype("Int64")
rm_set = set(rm_no2024["rm_id"].dropna().astype(int).tolist())

sub_m = submission.merge(mapping[["ID","rm_id"]], on="ID", how="left")
to_zero = sub_m["rm_id"].isin(rm_set)

if "predicted_weight" not in sub_m.columns:
    sub_m["predicted_weight"] = 0.0  # if your file only had ID

sub_m.loc[to_zero, "predicted_weight"] = 0.0

out = sub_m[["ID","predicted_weight"]]
out.to_csv(out_path, index=False)

affected = sub_m.loc[to_zero, ["ID","rm_id"]].drop_duplicates().sort_values(["rm_id","ID"])

print("Saved:", out_path)
